In [21]:
import pandas as pd 
import networkx as nx 
import numpy as np
import omnipath as op 

/home/exacloud/gscratch/mcweeney_lab/evans/external/miniforge3/envs/lincs-gsnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ppi = pd.read_csv('/home/exacloud/gscratch/mcweeney_lab/evans/data/comppi--compartments--tax_hsapiens_loc_all.txt', sep='\t')
ppi.head() 

,Interactor A,Naming Convention A,Major Loc A With Loc Score,Minor Loc A,Loc Experimental System Type A,Loc Source DB A,Loc PubMed ID A,Taxonomy ID A,Interactor B,Naming Convention B,Major Loc B With Loc Score,Minor Loc B,Loc Experimental System Type B,Loc Source DB B,Loc PubMed ID B,Taxonomy ID B,Interaction Score,Interaction Experimental System Type,Interaction Source Database,Interaction PubMed ID
0,A0A0R4J2E4,UniProtKB/TrEmbl/P,cytosol:0.8|nucleus:0.7,GO:0005634|GO:0005813,SVM decision tree(Predicted)|annotated protein...,eSLDB|The Human Protein Atlas,17108361|16127175,9606,A0A024R0Y4,UniProtKB/TrEmbl/P,nucleus:0.8,GO:0016607,annotated protein expression (APE)(Experimental),The Human Protein Atlas,16127175,9606,0.560000,two-hybrid screening(Experimental),CCSB,25416956
1,A0A0R4J2E4,UniProtKB/TrEmbl/P,cytosol:0.8|nucleus:0.7,GO:0005634|GO:0005813,SVM decision tree(Predicted)|annotated protein...,eSLDB|The Human Protein Atlas,17108361|16127175,9606,A0A087WWR4,UniProtKB/TrEmbl/P,nucleus:0.8,GO:0016607,annotated protein expression (APE)(Experimental),The Human Protein Atlas,16127175,9606,0.560000,two-hybrid screening(Experimental),CCSB,25416956
2,A0A0R4J2E4,UniProtKB/TrEmbl/P,cytosol:0.8|nucleus:0.7,GO:0005634|GO:0005813,SVM decision tree(Predicted)|annotated protein...,eSLDB|The Human Protein Atlas,17108361|16127175,9606,Q08379,UniProtKB/Swiss-Prot/P,cytosol:0.958|extracellular:0.8|membrane:0.8|n...,GO:0016020|GO:0005794|GO:0000137|GO:0000139|GO...,experimental(Experimental)|experimental(Experi...,eSLDB|eSLDB|GO|GO|GO|GO|GO|GO|GO|GO|GO|GO|LOCA...,17108361|17108361|10802651|10802651|10802651|1...,9606,0.880864,two-hybrid screening(Experimental),CCSB,25416956
3,A0A0R4J2E4,UniProtKB/TrEmbl/P,cytosol:0.8|nucleus:0.7,GO:0005634|GO:0005813,SVM decision tree(Predicted)|annotated protein...,eSLDB|The Human Protein Atlas,17108361|16127175,9606,Q5I0X7,UniProtKB/Swiss-Prot/P,cytosol:0.9099999999999999|mitochondrion:0.8|n...,GO:0005737|GO:0005634|GO:0005737|GO:0005739|GO...,SVM decision tree(Predicted)|PAML algorithm(Pr...,eSLDB|PA-GOSUB|PA-GOSUB|The Human Protein Atla...,17108361|15608166|15608166|16127175|16127175,9606,0.906976,two-hybrid screening(Experimental),CCSB,25416956
4,A0A0R4J2E4,UniProtKB/TrEmbl/P,cytosol:0.8|nucleus:0.7,GO:0005634|GO:0005813,SVM decision tree(Predicted)|annotated protein...,eSLDB|The Human Protein Atlas,17108361|16127175,9606,B7WNQ9,UniProtKB/TrEmbl/P,nucleus:0.7,GO:0005634,SVM decision tree(Predicted),eSLDB,17108361,9606,0.490000,two-hybrid screening(Experimental),CCSB,25416956


In [6]:
ppi[['Interactor A', 'Interactor B', 'Major Loc A With Loc Score', 'Major Loc B With Loc Score']].head()

,Interactor A,Interactor B,Major Loc A With Loc Score,Major Loc B With Loc Score
0,A0A0R4J2E4,A0A024R0Y4,cytosol:0.8|nucleus:0.7,nucleus:0.8
1,A0A0R4J2E4,A0A087WWR4,cytosol:0.8|nucleus:0.7,nucleus:0.8
2,A0A0R4J2E4,Q08379,cytosol:0.8|nucleus:0.7,cytosol:0.958|extracellular:0.8|membrane:0.8|n...
3,A0A0R4J2E4,Q5I0X7,cytosol:0.8|nucleus:0.7,cytosol:0.9099999999999999|mitochondrion:0.8|n...
4,A0A0R4J2E4,B7WNQ9,cytosol:0.8|nucleus:0.7,nucleus:0.7


In [14]:
edges = {'source':[], 'target':[], 'compartment':[]}
compartments = {'nucleus':set(), 'cytoplasm':set(), 'mitochondrion':set(), 
                'extracellular':set(), 'membrane':set(), 'cytosol':set(), 'secretory-pathway':set()}
all_nodes = set()

for i,row in ppi[['Interactor A', 'Interactor B', 'Major Loc A With Loc Score', 'Major Loc B With Loc Score']].iterrows():
    if i % 1000 == 0:
        print(f'Processing row {i} of {len(ppi)}', end='\r')

    all_nodes.add(row['Interactor A'])
    all_nodes.add(row['Interactor B'])

    A_locs = set([x.split(':')[0] for x in row['Major Loc A With Loc Score'].split('|')])
    B_locs = set([x.split(':')[0] for x in row['Major Loc B With Loc Score'].split('|')])

    overlap_locs = A_locs & B_locs 

    for loc in overlap_locs:
        compartments[loc].add(row['Interactor A'])
        compartments[loc].add(row['Interactor B'])
        edges['source'].append(loc + '__' + row['Interactor A'])
        edges['target'].append(loc + '__' + row['Interactor B'])
        edges['compartment'].append(loc)

# add cross compartment edges 
for loc1 in compartments:
    for loc2 in compartments:
        if loc1 == loc2:
            continue
        
        overlap = compartments[loc1] & compartments[loc2]
        for node in overlap:
            edges['source'].append(loc1 + '__' + node)
            edges['target'].append(loc2 + '__' + node)
            edges['compartment'].append(loc1 + '-' + loc2)

In [15]:
edge_df = pd.DataFrame(edges)
edge_df.groupby('compartment').count() 

,source,target
compartment,,
cytosol,398478,398478
cytosol-extracellular,6290,6290
cytosol-membrane,7982,7982
cytosol-mitochondrion,2044,2044
cytosol-nucleus,9960,9960
cytosol-secretory-pathway,4967,4967
extracellular,134465,134465
extracellular-cytosol,6290,6290
extracellular-membrane,8073,8073


In [38]:
edge_df2 = edge_df.copy().rename(columns={'source':'target', 'target':'source'})
edge_df = pd.concat([edge_df, edge_df2])

G = nx.from_pandas_edgelist(edge_df, source='source', target='target', create_using=nx.DiGraph)
print('# nodes:', len(G.nodes))
print('# edges:', len(G.edges))
print('density:', nx.density(G))
print('avg degree:', np.mean(list(dict(G.degree()).values())))
    

# nodes: 59917
# edges: 2441586
density: 0.0006801099889896759
avg degree: 81.49894020061085


In [41]:
tfs = op.interactions.CollecTRI().get(genesymbols=True)
tfs = tfs[['source', 'target']]
tfs = tfs.assign(source = ['nucleus__' + x for x in tfs['source']])
tfs = tfs.assign(target = ['rna__' + x for x in tfs['target']])

# add translation edges rna -> nucleus 
rnas = tfs['target'].values.tolist()
translation_edges = pd.DataFrame({
    'source': rnas, 
    'target': ['nucleus__' + x.split('__')[1] for x in rnas]
})

tfs = pd.concat([tfs, translation_edges])

# ad to G 
edge_df = pd.concat([edge_df, tfs])
G = nx.from_pandas_edgelist(edge_df, source='source', target='target', create_using=nx.DiGraph)

print('# nodes:', len(G.nodes))
print('# edges:', len(G.edges))
print('density:', nx.density(G))
print('avg degree:', np.mean(list(dict(G.degree()).values())))


# nodes: 69264
# edges: 2512755
density: 0.0005237707952360797
avg degree: 72.55587318087318


In [36]:
# see make_bio_network.py load_targetome() for context
DATA = '/home/exacloud/gscratch/mcweeney_lab/evans/data'
META = '/home/exacloud/gscratch/mcweeney_lab/evans/lincs-modeling/outputs/lincs-traj/runs/exp/default_v03/output/predict_grid'
MAX_DTI_KD = 100.0  # nM; make_bio_network default is 1000

drugs = pd.read_csv(f'{META}/pert_ids.csv')['pert_id'].astype(str).tolist()
clue_mapping = (
    pd.read_csv(f'{DATA}/compoundinfo_beta.txt', sep='\t')[['inchi_key', 'pert_id']]
    .assign(pert_id=lambda x: x['pert_id'].astype(str))
    .drop_duplicates()
)

tge = pd.read_csv(f'{DATA}/targetome_extended-01-23-25.csv')
tge = tge[
    tge['assay_type'].isin(['Kd', 'Ki'])
    & tge['assay_relation'].isin(['<', '<=', '='])
    & (tge['assay_value'] <= MAX_DTI_KD)
]
tge = tge.merge(clue_mapping, on='inchi_key', how='inner')

n_drugs_meta = len(drugs)
tge = tge[tge['pert_id'].isin(drugs)]
n_drugs_mapped = tge['pert_id'].nunique()
print(f'INCHI_KEY -> PERT_ID: {n_drugs_mapped} / {n_drugs_meta} meta drugs')

# keep uniprot accessions; do not map to gene / PROTEIN__ func nodes
dti = (
    tge[['pert_id', 'uniprot_id']]
    .assign(uniprot_id=lambda x: x['uniprot_id'].astype(str).str.strip())
    .drop_duplicates()
)
print(
    f'DTI: {len(dti)} edges from {dti["pert_id"].nunique()} drugs, '
    f'{dti["uniprot_id"].nunique()} uniprot targets'
)
dti.head()


INCHI_KEY -> PERT_ID: 341 / 341 meta drugs
DTI: 1940 edges from 341 drugs, 565 uniprot targets


,pert_id,uniprot_id
28,BRD-K43887077,P14416
33,BRD-K43887077,P21917
37,BRD-K43887077,P35462
98,BRD-K97530723,P16083
100,BRD-K97530723,P48039


In [37]:
dti_targets = set(dti['uniprot_id'])
overlap = dti_targets & all_nodes
print(f'{len(overlap)} / {len(dti_targets)} DTI targets in ComPPI graph ({len(all_nodes)} nodes)')

dti_edges = pd.DataFrame({
    'src': 'DRUG__' + dti['pert_id'].astype(str),
    'dst': dti['uniprot_id'],
})
dti_edges.head()


556 / 565 DTI targets in ComPPI graph (23447 nodes)


,src,dst
28,DRUG__BRD-K43887077,P14416
33,DRUG__BRD-K43887077,P21917
37,DRUG__BRD-K43887077,P35462
98,DRUG__BRD-K97530723,P16083
100,DRUG__BRD-K97530723,P48039
